<a href="https://colab.research.google.com/github/zeenatyousef/Language-Tutor/blob/main/Language_Tutor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install groq gTTS
!apt-get install -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
from groq import Groq
from google.colab import userdata

# Read API key securely from Colab Secrets
API_KEY = userdata.get('API')

client = Groq(api_key=API_KEY)
print("✅ Groq connected securely!")

✅ Groq connected securely!


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
from gtts import gTTS
import tempfile, os, base64
from google.colab import output

# ── Languages ─────────────────────────────────────────────────────
LANGUAGES = [
    "English", "Spanish", "French", "German", "Italian",
    "Portuguese", "Arabic", "Chinese", "Japanese", "Korean",
    "Turkish", "Hindi", "Russian", "Dutch", "Polish" , "Urdu"
]

# Language code map for gTTS
LANG_CODES = {
    "English": "en", "Spanish": "es", "French": "fr",
    "German": "de", "Italian": "it", "Portuguese": "pt",
    "Arabic": "ar", "Chinese": "zh", "Japanese": "ja",
    "Korean": "ko", "Turkish": "tr", "Hindi": "hi",
    "Russian": "ru", "Dutch": "nl", "Polish": "pl" , "Urdu": "ur"
}

# ── UI Widgets ────────────────────────────────────────────────────
title = widgets.HTML("<h2>🎓 AI Language Tutor — Powered by Groq</h2>")

user_lang_dropdown = widgets.Dropdown(
    options=LANGUAGES, value="English",
    description="🗣️ Your Language:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="350px")
)

output_lang_dropdown = widgets.Dropdown(
    options=LANGUAGES, value="Urdu",
    description="📖 Learn Language:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="350px")
)

tts_toggle = widgets.Checkbox(
    value=True,
    description="🔊 Tutor Speaks (Text-to-Speech)",
    style={"description_width": "initial"}
)

start_btn = widgets.Button(
    description="Start Session", button_style="success",
    icon="play", layout=widgets.Layout(width="200px", margin="10px 0")
)

text_input = widgets.Text(
    placeholder="Type your message here...",
    layout=widgets.Layout(width="460px"), disabled=True
)

send_btn = widgets.Button(
    description="Send", button_style="primary",
    icon="paper-plane", disabled=True,
    layout=widgets.Layout(width="100px")
)

mic_btn = widgets.Button(
    description="🎤 Speak", button_style="warning",
    disabled=True, layout=widgets.Layout(width="120px")
)

chat_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ccc", padding="10px",
        min_height="300px", max_height="500px",
        overflow_y="auto", width="680px"
    )
)

audio_output = widgets.Output()

# ── State ─────────────────────────────────────────────────────────
chat_history = []

SYSTEM_PROMPT_TEMPLATE = """
You are an expert, friendly AI Language Tutor.
The student speaks {user_lang} and is learning {output_lang}.
For EVERY student message, always reply with all 4 sections:

1. 💬 CONVERSATION REPLY — Respond in {output_lang}, then translate to {user_lang}
2. ✅ GRAMMAR CHECK — Correct any mistakes. If none, say "Perfect!"
3. 📖 VOCABULARY BOOST — 1-2 new words with definition + example
4. 🔊 PRONUNCIATION TIP — Phonetic tip for one tricky word

Be encouraging, patient, and beginner-friendly.
"""

# ── Text-to-Speech ────────────────────────────────────────────────
def speak(text, lang_code):
    if not tts_toggle.value:
        return
    try:
        # Extract only the CONVERSATION REPLY part to speak
        lines = text.split("\n")
        speak_text = ""
        capture = False
        for line in lines:
            if "CONVERSATION REPLY" in line:
                capture = True
                continue
            if capture and line.startswith("2.") or (capture and "GRAMMAR" in line):
                break
            if capture and line.strip():
                speak_text += line.strip() + " "

        if not speak_text:
            speak_text = text[:300]  # fallback

        tts = gTTS(text=speak_text, lang=lang_code, slow=False)
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
            tts.save(f.name)
            fname = f.name

        with audio_output:
            clear_output()
            display(Audio(fname, autoplay=True))
    except Exception as e:
        with audio_output:
            clear_output()
            print(f"⚠️ Audio error: {e}")

# ── Speech-to-Text via Groq Whisper ──────────────────────────────
def on_mic(b):
    with chat_output:
        print("\n🎤 Recording... (speak now, runs for 5 seconds)")

    # Record audio using JavaScript in Colab
    output.eval_js("""
    async function recordAudio() {
        const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
        const recorder = new MediaRecorder(stream);
        const chunks = [];
        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();
        await new Promise(r => setTimeout(r, 5000));
        recorder.stop();
        await new Promise(r => recorder.onstop = r);
        const blob = new Blob(chunks, { type: 'audio/webm' });
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        await new Promise(r => reader.onload = r);
        return reader.result.split(',')[1];
    }
    recordAudio().then(data => {
        google.colab.kernel.invokeFunction('notebook.process_audio', [data], {});
    });
    """)

def process_audio(audio_b64):
    try:
        audio_data = base64.b64decode(audio_b64)
        with tempfile.NamedTemporaryFile(suffix=".webm", delete=False) as f:
            f.write(audio_data)
            fname = f.name

        with open(fname, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(
                model="whisper-large-v3",
                file=("audio.webm", audio_file, "audio/webm"),
            )

        text_input.value = transcription.text
        with chat_output:
            print(f"🎤 Transcribed: {transcription.text}")

    except Exception as e:
        with chat_output:
            print(f"⚠️ Transcription error: {e}")

output.register_callback("notebook.process_audio", process_audio)

# ── Start Session ────────────────────────────────────────────────-
def on_start(b):
    global chat_history
    chat_history = []

    user_lang = user_lang_dropdown.value
    output_lang = output_lang_dropdown.value
    lang_code = LANG_CODES.get(output_lang, "en")

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        user_lang=user_lang, output_lang=output_lang
    )
    chat_history.append({"role": "system", "content": system_prompt})
    chat_history.append({
        "role": "user",
        "content": f"Greet me warmly in {output_lang} (with {user_lang} translation) and ask a simple question to start."
    })

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=chat_history,
        temperature=0.7, max_tokens=512,
    )
    greeting = response.choices[0].message.content
    chat_history.append({"role": "assistant", "content": greeting})

    text_input.disabled = False
    send_btn.disabled = False
    mic_btn.disabled = False

    with chat_output:
        clear_output()
        print(f"📚 Session: {user_lang} → Learning {output_lang}\n")
        print("─" * 60)
        print(f"🤖 Tutor:\n{greeting}")
        print("─" * 60)

    speak(greeting, lang_code)

# ── Send Message ──────────────────────────────────────────────────
def on_send(b):
    user_input = text_input.value.strip()
    if not user_input:
        return

    text_input.value = ""
    lang_code = LANG_CODES.get(output_lang_dropdown.value, "en")

    chat_history.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=chat_history,
        temperature=0.7, max_tokens=1024,
    )
    reply = response.choices[0].message.content
    chat_history.append({"role": "assistant", "content": reply})

    with chat_output:
        print(f"\n👤 You: {user_input}")
        print(f"\n🤖 Tutor:\n{reply}")
        print("─" * 60)

    speak(reply, lang_code)

# ── Listeners ─────────────────────────────────────────────────────
start_btn.on_click(on_start)
send_btn.on_click(on_send)
mic_btn.on_click(on_mic)

# ── Display UI ────────────────────────────────────────────────────
input_row = widgets.HBox([text_input, send_btn, mic_btn])
display(widgets.VBox([
    title,
    user_lang_dropdown,
    output_lang_dropdown,
    tts_toggle,
    start_btn,
    chat_output,
    input_row,
    audio_output
]))

## 🖥️ What the UI Looks Like
```
🎓 AI Language Tutor — Powered by Groq

🗣️ Your Language:    [ English   ▼ ]
📖 Learn Language:   [ Spanish   ▼ ]
☑️  🔊 Tutor Speaks (Text-to-Speech)

        [ ▶ Start Session ]

┌──────────────────────────────────────────────┐
│ 📚 Session: English → Learning Spanish       │
│ ──────────────────────────────────────────── │
│ 🤖 Tutor:                                    │
│ ¡Hola! Welcome! (Hello! Welcome!)            │
│ ¿Cómo te llamas? (What is your name?)        │
└──────────────────────────────────────────────┘

[ Type your message... ] [Send] [🎤 Speak]
🔈 (audio plays automatically)
```